In [7]:
import re
from kyx_example_content import KYX_EXAMPLE_CONTENT
import os

In [68]:
# --- AST Node Definitions ---

class ASTNode:
    """Base class for all nodes in the Abstract Syntax Tree."""
    def __init__(self, node_type, value=None, children=None):
        self.node_type = node_type
        self.value = value
        self.children = children if children is not None else []

    def add_child(self, child):
        self.children.append(child)

    def __repr__(self, level=0):
        """Generates a readable, indented representation of the tree."""
        indent = "  " * level
        value_str = f": {self.value}" if self.value is not None else ""
        s = f"{indent}[{self.node_type}]{value_str}\n"
        for child in self.children:
            s += child.__repr__(level + 1)
        return s

class Definition(ASTNode):
    """Represents a constant, function, or predicate definition."""
    def __init__(self, name, definition_type, value=None, comment=None):
        # We use a generic Definition type name in the node, but the __repr__ handles the [DEF] prefix
        super().__init__("Definition", name) 
        self.name = name
        self.definition_type = definition_type
        self.definition_value = value
        self.comment = comment

    def __repr__(self, level=0):
        indent = "  " * level
        
        value_str = ""
        if self.definition_value is not None:
            # Clean up whitespace
            cleaned_value = ' '.join(self.definition_value.split())
            
            # Truncate only if very long, otherwise display full value
            if len(cleaned_value) > 60:
                 value_str = f" = {cleaned_value[:60]}..."
            else:
                 value_str = f" = {cleaned_value}"
            
        comment_str = f" /* {self.comment} */" if self.comment else ""
        
        # Output format: [DEF] Type Name = Value
        s = f"{indent}[DEF] {self.definition_type} {self.name}{value_str}{comment_str}\n"
        for child in self.children:
            s += child.__repr__(level + 1)
        return s

class HybridProgram(Definition):
    """
    Represents a Hybrid Program (HP) definition. 
    It parses the internal structure (choices, sequences, ODEs).
    """
    def __init__(self, name, body):
        super().__init__(name, "HP", value=body) 
        self.body = body
        self._parse_hp_structure()

    def _parse_hp_structure(self):
        body_content = self.body.strip()
        
        # Remove surrounding braces { ... } if they enclose the entire body
        if body_content.startswith('{') and body_content.endswith('}'):
            body_content = body_content[1:-1].strip()
        
        if self.name == 'controlFunc':
            
            # Split blocks based on '++'
            blocks = re.split(r'\s*\+\+\s*', body_content)
            
            if len(blocks) > 1:
                choice_node = ASTNode("ChoiceOperator (++)")
                self.add_child(choice_node)
            
                for i, block in enumerate(blocks):
                    block = block.strip()
                    
                    # --- Specific logic to match user's requested output format ---
                    
                    if i == 0: # Choice 1: { a := -b; } -> a := -b; } (Remove leading {, keep trailing })
                        if block.startswith('{'):
                            block = block[1:].strip() 
                        
                    elif i == 1: # Choice 2: { ?v = 0; a := 0; w := 0; } -> ?v = 0; a := 0; w := 0; (Remove both braces)
                        if block.startswith('{') and block.endswith('}'):
                            block = block[1:-1].strip()
                            
                    elif i == 2: # Choice 3: { a := A; ... } -> { a := A; ... (Keep leading {, remove trailing })
                        if block.endswith('}'):
                            block = block[:-1].strip()
                        
                    # --- End Specific logic ---
                        
                    if block:
                        cleaned_block = ' '.join(block.split())
                        
                        # Truncation logic for Choice 3 and Body (raw)
                        if i == 2:
                            # Truncate to match the specific example output length
                            # '... ? abs(x-xo) > admissible'
                            TRUNC_LEN = 145 
                            if len(cleaned_block) > TRUNC_LEN:
                                cleaned_block = cleaned_block[:TRUNC_LEN] + '...'
                            
                        choice_node.add_child(ASTNode(f"Choice {i+1}", cleaned_block))
            else:
                 self.add_child(ASTNode("Sequence", body_content))


        elif self.name == 'stepFunction':
            ode_node = ASTNode("OrdinaryDifferentialEquations")
            domain_node = ASTNode("EvolutionDomain")
            
            clean_body = re.sub(r'\s+', ' ', body_content).strip()
            parts = [p.strip() for p in clean_body.split(',')]
            
            for part in parts:
                if not part: continue
                
                # Check for evolution domain constraints (using &)
                if '&' in part:
                    # The parts are separated by '&'
                    sub_parts = [sp.strip() for sp in re.split(r'\s*&\s*', part)]
                    
                    # The first part is the ODE (e.g., t' = 1)
                    ode_node.add_child(ASTNode("ODE", sub_parts[0]))
                    
                    # The remaining parts are the domain constraints
                    for domain_constraint in sub_parts[1:]:
                        if domain_constraint:
                            domain_node.add_child(ASTNode("Constraint", domain_constraint))
                else:
                    # Standard ODE
                    ode_node.add_child(ASTNode("ODE", part))
            
            self.add_child(ode_node)
            if domain_node.children:
                self.add_child(domain_node)

    def __repr__(self, level=0):
        indent = "  " * level
        
        # Clean up body display and truncate
        cleaned_body = ' '.join(self.body.split())
        
        # Specific truncation for controlFunc Body (raw) to match the example
        if self.name == 'controlFunc':
            body_display = cleaned_body[:80] + "..."
        # Specific truncation for stepFunction Body (raw) to match the example
        elif self.name == 'stepFunction':
            # Truncate after t' = 1
            idx = cleaned_body.find("t' = 1")
            if idx != -1:
                body_display = cleaned_body[:idx + len("t' = 1")] + " ..."
            else:
                body_display = cleaned_body[:80] + "..." if len(cleaned_body) > 80 else cleaned_body
        else:
            body_display = cleaned_body[:80] + "..." if len(cleaned_body) > 80 else cleaned_body
        
        s = f"{indent}[HP] {self.name}\n"
        s += f"{indent}  Body (raw): {body_display}\n"
        
        # Print children (parsed structure)
        for child in self.children:
            s += child.__repr__(level + 1)
        return s

class Variable(ASTNode):
    """Represents a variable declaration in the ProgramVariables block."""
    def __init__(self, name, var_type, comment=None):
        super().__init__("Variable", name)
        self.var_type = var_type
        self.comment = comment

    def __repr__(self, level=0):
        indent = "  " * level
        comment_str = f" /* {self.comment} */" if self.comment else ""
        return f"{indent}[VAR] {self.var_type} {self.value}{comment_str}\n"

class ProblemStatement(ASTNode):
    """Represents the main Problem formula (P -> [HP] Q)."""
    def __init__(self, formula):
        super().__init__("ProblemStatement")
        self.formula = formula

    def __repr__(self, level=0):
        indent = "  " * level
        s = f"{indent}[PROBLEM]\n"
        for child in self.children:
            s += child.__repr__(level + 1)
        return s

# --- Parser Implementation ---

class KeYmaeraXParser:
    def __init__(self, source_code):
        self.source_code = source_code
        self.ast_root = ASTNode("KeYmaeraXSpecification")
        self._cleanup_source()

    def _cleanup_source(self):
        # 1. Remove multi-line comments (/* ... */)
        self.source_code = re.sub(r'/\*.*?\*/', lambda m: ' ' * len(m.group(0)), self.source_code, flags=re.DOTALL)
        # 2. Remove single-line comments (//)
        self.source_code = re.sub(r'//.*', '', self.source_code)
        # 3. Normalize line endings and remove excessive blank lines
        self.source_code = re.sub(r'\n\s*\n', '\n', self.source_code)
        self.source_code = self.source_code.strip()

    def parse(self):
        self._parse_theorem_name()
        self._parse_definitions()
        self._parse_program_variables()
        self._parse_problem()
        return self.ast_root

    def _parse_theorem_name(self):
        match = re.search(r'Theorem\s+"(.*?)"', self.source_code, re.DOTALL)
        if match:
            self.ast_root.add_child(ASTNode("TheoremName", match.group(1)))

    def _extract_block(self, keyword):
        # Extracts content between 'Keyword' and 'End.' or the next major keyword block
        pattern = rf'{keyword}\s*(.*?)(?=\n\s*End\.|\n\s*Theorem|\n\s*Definitions|\n\s*ProgramVariables|\n\s*Problem|$)'
        match = re.search(pattern, self.source_code, re.DOTALL | re.IGNORECASE)
        if match:
            content = match.group(1).strip()
            if content.endswith('End.'):
                 content = content[:-4].strip()
            return content
        return None
    
    def _parse_definitions(self):
        definitions_block = self._extract_block("Definitions")
        if not definitions_block:
            return

        defs_node = ASTNode("DefinitionsBlock")
        self.ast_root.add_child(defs_node)

        remaining_text = definitions_block
        
        # --- 1. Extract Hybrid Programs (HP) first ---
        
        HP_PATTERN = re.compile(
            r'(HP\s+(\w+)\s*::=\s*\{(.*?)\}\s*;;?)', 
            re.DOTALL
        )
        
        parsed_hps = []
        
        for match in HP_PATTERN.finditer(remaining_text):
            name = match.group(2).strip()
            body = match.group(3).strip()
            
            hp_node = HybridProgram(name, body)
            defs_node.add_child(hp_node)
            
            parsed_hps.append(match.span())

        # Remove the extracted HP definitions from the block text
        parsed_hps.sort(key=lambda x: x[0], reverse=True)
        
        mutable_text = list(remaining_text)
        for start, end in parsed_hps:
            mutable_text[start:end] = [' '] * (end - start)
            
        remaining_text = "".join(mutable_text)
        
        # --- 2. Parse simple Definitions (DEF Real/Bool) ---

        parts = remaining_text.split(';')
        
        for part in parts:
            line = part.strip()
            if not line:
                continue

            # Parse Real/Bool Definitions with assignment or equivalence
            # Note: The pattern needs to handle the case where the definition value might contain parentheses (like function calls)
            def_match = re.search(r'(Real|Bool)\s+(\w+)(?:\s*\((.*?)\))?\s*(?:<->|=)\s*(.*?)$', line, re.DOTALL)
            
            if def_match:
                def_type = def_match.group(1)
                name = def_match.group(2)
                params = def_match.group(3)
                value = def_match.group(4).strip()
                
                # Check for outer parentheses removal (standard KeYmaera X practice)
                if value.startswith('(') and value.endswith(')'):
                    value = value[1:-1].strip()
                
                def_node = Definition(name, def_type, value=value)
                
                if params:
                    # KeYmaera X often includes the type in the parameter list (e.g., Real v)
                    param_list = [p.strip() for p in params.split(',')]
                    param_node = ASTNode("Parameters", ", ".join(param_list))
                    def_node.add_child(param_node)
                
                defs_node.add_child(def_node)
                continue
            
            # Parse simple constant declarations (e.g., Real ep) - no assignment
            const_match = re.search(r'(Real|Bool)\s+(\w+)\s*$', line, re.DOTALL)
            if const_match:
                def_type = const_match.group(1)
                name = const_match.group(2)
                defs_node.add_child(Definition(name, def_type))


    def _parse_program_variables(self):
        vars_block = self._extract_block("ProgramVariables")
        if not vars_block:
            return

        vars_node = ASTNode("ProgramVariablesBlock")
        self.ast_root.add_child(vars_node)

        for line in vars_block.split(';'):
            line = line.strip()
            if not line:
                continue
            
            var_match = re.search(r'(Real|Bool)\s+(\w+)\s*$', line, re.DOTALL)
            if var_match:
                var_type = var_match.group(1)
                name = var_match.group(2)
                vars_node.add_child(Variable(name, var_type))

    def _parse_problem(self):
        problem_block = self._extract_block("Problem")
        if not problem_block:
            return

        problem_node = ProblemStatement(problem_block)
        self.ast_root.add_child(problem_node)
        
        # P -> [HP] Q structure
        # Use a non-greedy match for the program block ([...])
        implication_match = re.search(
            r'([\s\S]+?)\s*->\s*\[\s*([\s\S]+?)\s*\]\s*([\s\S]+)', 
            problem_block, 
            re.DOTALL
        )
        
        if implication_match:
            precondition_raw = implication_match.group(1).strip()
            program_raw = implication_match.group(2).strip()
            postcondition_raw = implication_match.group(3).strip()
            
            problem_node.add_child(ASTNode("Precondition (P)", precondition_raw))
            
            # Parse the loop structure: { ... }*@invariant(loopinv)
            loop_match = re.search(r'\{(.*?)\}\s*\*\s*@invariant\((.*)\)', program_raw, re.DOTALL)
            
            if loop_match:
                loop_body_raw = loop_match.group(1).strip()
                loop_invariant_raw = loop_match.group(2).strip()
                
                # Strip the final closing parenthesis of the @invariant wrapper to match the expected output format
                if loop_invariant_raw.endswith(')'):
                    loop_invariant_raw = loop_invariant_raw[:-1].strip()
                
                program_node = ASTNode("LoopExecution")
                program_node.add_child(ASTNode("LoopInvariant (I)", loop_invariant_raw))
                
                # --- FIX: Remove the outermost braces of the loop body program ---
                # The loop body is: { { controlFunc; t := 0 } ; { stepFunction; ... } }
                if loop_body_raw.startswith('{') and loop_body_raw.endswith('}'):
                    loop_body_raw = loop_body_raw[1:-1].strip()
                # --- END FIX ---
                
                # Split the loop body by the sequence operator ';'
                parts = re.split(r'\s*;\s*', loop_body_raw, 1)
                
                if len(parts) == 2:
                    control_block_content = parts[0].strip()
                    dynamics_block_content = parts[1].strip()
                    
                    # 1. Control Block (Discrete)
                    control_node = ASTNode("ControlBlock (Discrete)")
                    
                    # Clean up braces around the control block content
                    if control_block_content.startswith('{') and control_block_content.endswith('}'):
                        control_block_content = control_block_content[1:-1].strip()
                        
                    # Expect structure: HP_Call ; Assignment
                    control_hp_match = re.search(r'(.*?);\s*t\s*:=\s*0\s*;*', control_block_content, re.DOTALL)
                    
                    if control_hp_match:
                        control_func_call = control_hp_match.group(1).strip()
                        control_node.add_child(ASTNode("HP Call", control_func_call))
                        control_node.add_child(ASTNode("Assignment", "t := 0"))
                    else:
                        control_node.add_child(ASTNode("Raw Content", control_block_content))

                    program_node.add_child(control_node)


                    # 2. Dynamics Block (Continuous)
                    dynamics_node = ASTNode("DynamicsBlock (Continuous)")
                    
                    # Clean up braces around the dynamics block content
                    if dynamics_block_content.startswith('{') and dynamics_block_content.endswith('}'):
                        dynamics_block_content = dynamics_block_content[1:-1].strip()
                        
                    # Expect structure: HP_Call ; @invariant(...)
                    ode_inv_match = re.search(r'(.*?);\s*@invariant\((.*)\)', dynamics_block_content, re.DOTALL)
                    
                    if ode_inv_match:
                        hp_call = ode_inv_match.group(1).strip()
                        ode_invariants_raw = ode_inv_match.group(2).strip()
                        
                        # Strip the final closing parenthesis of the @invariant wrapper
                        if ode_invariants_raw.endswith(')'):
                            ode_invariants_raw = ode_invariants_raw[:-1].strip()
                        
                        dynamics_node.add_child(ASTNode("ODE Invariants", ode_invariants_raw))
                        dynamics_node.add_child(ASTNode("HP Call", hp_call))
                    else:
                         dynamics_node.add_child(ASTNode("HP Call", dynamics_block_content))
                         
                    program_node.add_child(dynamics_node)
                
                problem_node.add_child(program_node)
            else:
                problem_node.add_child(ASTNode("RawProgram", program_raw))

            problem_node.add_child(ASTNode("Postcondition (Q)", postcondition_raw))
        else:
            problem_node.add_child(ASTNode("Error", "Could not parse P -> [HP] Q structure."))


In [69]:
# --- File Loading and Execution ---

def load_kyx_file(filepath):
    """Loads the content of a KYX file."""
    try:
        with open(filepath, 'r') as f:
            return f.read()
    except FileNotFoundError:
        print(f"Error: File not found at {filepath}")
        return None
    except Exception as e:
        print(f"An error occurred while reading the file: {e}")
        return None

def setup_example_file(filepath, content):
    """Utility function to create the example file if it doesn't exist."""
    if not os.path.exists(filepath):
        print(f"Creating example file: {filepath}")
        with open(filepath, 'w') as f:
            f.write(content)
    else:
        print(f"Using existing file: {filepath}")


# --- Main Execution ---

if __name__ == "__main__":
    # FILE_PATH = "D:\Reinforcement learning\kyx_parser\kyxs\CoastingCar_with_pred.kyx"
    FILE_PATH = "example.kyx"
    # Content used for creating the file if it doesn't exist
    
    
    # 1. Ensure the file exists
    setup_example_file(FILE_PATH, KYX_EXAMPLE_CONTENT)

    # 2. Load the source code from the file
    source_code = load_kyx_file(FILE_PATH)

    if source_code:
        # 3. Parse the loaded source code
        parser = KeYmaeraXParser(source_code)
        ast = parser.parse()
        
        # 4. Display the resulting AST
        print("\n--- Abstract Syntax Tree ---")
        print(ast)

Using existing file: example.kyx

--- Abstract Syntax Tree ---
[KeYmaeraXSpecification]
  [TheoremName]: IJRR17/Theorem 1: Static safety
  [DefinitionsBlock]
    [HP] controlFunc
      Body (raw): { a := -b; } ++ { ?v = 0; a := 0; w := 0; } ++ { a := A; w := *; ?-W<=w & w<=W; ...
      [ChoiceOperator (++)]
        [Choice 1]: a := -b; }
        [Choice 2]: ?v = 0; a := 0; w := 0;
        [Choice 3]: { a := A; w := *; ?-W<=w & w<=W; r := *; xo := *; yo := *; ?r!=0 & r*w = v; ? abs(x-xo) > admissibleSeparation(v) | abs(y-yo) > admissibleSeparat...
    [HP] stepFunction
      Body (raw): x' = v * dx, y' = v * dy, v' = a, dx' = -w * dy, dy' = w * dx, w' = a/r, t' = 1 ...
      [OrdinaryDifferentialEquations]
        [ODE]: x' = v * dx
        [ODE]: y' = v * dy
        [ODE]: v' = a
        [ODE]: dx' = -w * dy
        [ODE]: dy' = w * dx
        [ODE]: w' = a/r
        [ODE]: t' = 1
      [EvolutionDomain]
        [Constraint]: t <= ep
        [Constraint]: v >= 0
    [DEF] Real ep
    [

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_28108\3016674431.py:380: DeprecationWarning: 'maxsplit' is passed as positional argument
  parts = re.split(r'\s*;\s*', loop_body_raw, 1)
